# Pan-European Grid Resilience: System Overview

## The Problem

Europe's power grid is undergoing rapid decarbonisation. Wind now supplies 20–30% of electricity in the Nordic and Baltic regions, and that share is growing. This creates a class of risk that traditional grid adequacy studies were never designed to handle: **correlated wind droughts**.

A correlated wind drought occurs when a large, slow-moving high-pressure system parks over northern Europe and suppresses wind generation simultaneously across Denmark, Sweden, southern Norway, and Finland for 48–96 hours. During those windows, every utility-scale wind farm in the region is producing near zero. Thermal and hydro reserves run down, interconnector flows hit physical limits, and spot prices spike to 400+ EUR/MWh.

### Why deterministic models fail here

Traditional capacity adequacy methods (N-1 contingency analysis, LOLE tables) treat weather as a scalar input — one region, one season, one scenario. They miss three things:

1. **Spatial correlation**: Wind speed at SE3 (Stockholm region) is strongly correlated with DK2 (Copenhagen region) when the same synoptic system covers both. A model that treats zones as independent underestimates the probability of simultaneous multi-zone shortfall by orders of magnitude.
2. **Non-Gaussian tail behaviour**: The distribution of wind speed is left-skewed. Droughts are rare but heavy-tailed — the kind of events a Gaussian mean model smooths away.
3. **Probabilistic uncertainty**: A point forecast says "wind will be 6 m/s". A probabilistic forecast says "there is a 12% chance wind drops below 3 m/s in at least 4 zones simultaneously within 48 hours". Only the latter is actionable for reserve procurement.

### What this system does

We build a **Bayesian spatio-temporal Gaussian Process** (specifically, a Hilbert Space GP approximation) that:

- Learns the full joint spatial covariance across 9 Nordic zones from historical or synthetic hourly wind observations
- Samples the posterior distribution over latent wind speed trajectories via MCMC
- Produces P10/P50/P90 probabilistic forecasts and computes **P(shortfall)** — the probability that wind speed in any zone falls below the 3 m/s turbine cut-in threshold at each 24h/48h/72h forecast horizon
- Serves those forecasts through a REST API consumed by a risk dashboard

## System Architecture

The platform is layered into five tiers: **data ingestion → ETL → inference → serving → observability**.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  DATA SOURCES                                                               │
│  ERA5 reanalysis (ECMWF CDS API) ─────────────────────────────────────────┐│
│  ENTSO-E Transparency Platform (load, cross-border flows) ────────────────┤│
│  Synthetic generator (Matern covariance, for MVP / CI) ───────────────────┘│
│                                    │                                        │
│                            Bronze bucket (Ceph RGW)                         │
│                            NetCDF / XML / Parquet                           │
│                                    │                                        │
│  ETL                               ▼                                        │
│  PySpark (SparkApplication on K8s) ──── spatial join + schema normalise    │
│                                    │                                        │
│                            Silver bucket (Ceph RGW)                         │
│                            grid_snapshots.parquet                           │
│                            columns: timestamp | region_id | wind_speed_mps  │
│                                    │                                        │
│  INFERENCE                         ▼                                        │
│  Dagster (MVP) / KubeRay RayJob (production)                                │
│    └── PyMC HSGP model (Matern 5/2, separable spatial × temporal)          │
│    └── Nutpie NUTS sampler (Rust, ~10× faster than PyMC default)           │
│    └── ArviZ diagnostics (R-hat, ESS, divergences, CRPS)                   │
│    └── MLflow experiment tracking                                           │
│                                    │                                        │
│                            Gold bucket (Ceph RGW)                           │
│                            tail_risk_forecasts.parquet                      │
│                            hsgp_model.pkl  +  hsgp_model_idata.nc          │
│                                    │                                        │
│  SERVING                           ▼                                        │
│  FastAPI   GET /forecast/{region_id}?horizon_h=24  →  {p_shortfall}        │
│            GET /correlations                       →  {correlation_matrix}  │
│  Streamlit risk dashboard  (P10/P50/P90 choropleth, Phase 2)               │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Component responsibilities

| Layer | Technology | Purpose |
|---|---|---|
| Ingestion | Python scripts + ECMWF CDS SDK | Pull ERA5 10 m wind NetCDF; ENTSO-E XML load |
| Storage | Ceph RGW (S3-compatible) | Immutable Bronze/Silver/Gold bucket hierarchy |
| ETL | PySpark on Kubernetes | Schema normalisation, spatial join, Parquet output |
| Orchestration | Dagster (MVP) / KubeRay (prod) | Asset DAG, scheduling, retries, IO managers |
| Model | PyMC v5 HSGP | Bayesian spatio-temporal GP inference |
| Sampler | Nutpie (Rust NUTS) | 5–10× faster than PyMC default NUTS |
| Diagnostics | ArviZ + MLflow | Convergence checks, experiment tracking |
| API | FastAPI + uvicorn | REST endpoints for forecast consumers |
| Streaming | Bytewax + Redpanda | Real-time spatial windowing (Phase 2) |

## Imports & Setup

All cells below run self-contained: if PostgreSQL is unavailable, we fall back to a small in-memory synthetic dataset.

In [ ]:
from __future__ import annotations

import sys
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.linalg import cholesky

# Make sure the repo root is on the path so we can import pipeline modules
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." , ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# PyMC / ArviZ — lazy-imported later so the notebook loads even without them
try:
    import pymc as pm
    import arviz as az
    HAS_PYMC = True
    print(f"PyMC {pm.__version__} | ArviZ {az.__version__}")
except ImportError:
    HAS_PYMC = False
    print("PyMC/ArviZ not installed — model cells will be skipped")

rng = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
print("Setup complete")

## Synthetic Data Generator

### Why synthetic data?

Real ERA5 data requires an ECMWF account and ~minutes of download time. For development, testing, and CI, `generator/generate.py` produces statistically realistic correlated wind speed series using the exact same Matern covariance structure that the HSGP model will later try to *recover*. This gives us a ground truth to validate against.

### How it works

**Step 1 — Spatial covariance matrix.** We compute the pairwise great-circle distances between the 9 zone centroids and evaluate a **Matern 3/2** kernel:

$$K(r) = \sigma^2 \left(1 + \frac{\sqrt{3}\,r}{\ell}\right) \exp\!\left(-\frac{\sqrt{3}\,r}{\ell}\right)$$

where $r$ is distance in km, $\ell = 100$ km is the spatial length scale, and $\sigma^2 = 4$ (m/s)² is the marginal variance. Nearby zones get correlation near 1; zones 500+ km apart approach 0.

**Step 2 — Cholesky sampling.** We factorise $K = LL^\top$ and draw correlated wind anomalies as $\mathbf{w} = \bar{w} + (L\,\mathbf{z}^\top)^\top$ where $\mathbf{z} \sim \mathcal{N}(0, I)$ and $\bar{w} = 7$ m/s is the background mean.

**Step 3 — Drought injection.** We inject 5–10 correlated drought events per year: at a random timestep, 2–9 zones simultaneously have their wind speed scaled to 40% of the original value. These events simulate the synoptic-scale blocking high-pressure systems we care about most.

**Step 4 — Storage.** Data is written to PostgreSQL partitioned tables (`weather_obs`, `grid_demand`, `spot_prices`) for 2019–2024 at 1-hour resolution (9 zones × ~52,600 hours ≈ 474,000 rows).

In [ ]:
# ── Reproduce the generator's covariance matrix in-memory ──────────────────
from math import radians, sin, cos, sqrt, atan2

ZONE_CENTROIDS = {
    "SE1": (62.0, 14.5),
    "SE2": (62.5, 17.5),
    "SE3": (59.5, 18.0),
    "SE4": (56.0, 14.0),
    "NO1": (59.5, 10.5),
    "NO2": (64.0, 11.5),
    "DK1": (56.5,  9.5),
    "DK2": (55.5, 12.0),
    "FI":  (61.5, 25.0),
}
ZONES = list(ZONE_CENTROIDS.keys())
N_ZONES = len(ZONES)


def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlam = radians(lon2 - lon1)
    a = sin(dphi / 2)**2 + cos(phi1) * cos(phi2) * sin(dlam / 2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


def matern32_cov(zone_ids, lengthscale_km=100.0, sigma2=4.0):
    n = len(zone_ids)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            lat1, lon1 = ZONE_CENTROIDS[zone_ids[i]]
            lat2, lon2 = ZONE_CENTROIDS[zone_ids[j]]
            d = haversine_km(lat1, lon1, lat2, lon2)
            D[i, j] = D[j, i] = d
    r = D / lengthscale_km
    K = sigma2 * (1 + np.sqrt(3) * r) * np.exp(-np.sqrt(3) * r)
    np.fill_diagonal(K, sigma2)
    return K, D


K_true, D_km = matern32_cov(ZONES, lengthscale_km=100.0, sigma2=4.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

im0 = axes[0].imshow(D_km, cmap="YlOrRd")
axes[0].set_title("Pairwise great-circle distances (km)", fontsize=12)
axes[0].set_xticks(range(N_ZONES)); axes[0].set_xticklabels(ZONES, rotation=45, ha="right")
axes[0].set_yticks(range(N_ZONES)); axes[0].set_yticklabels(ZONES)
plt.colorbar(im0, ax=axes[0], label="km")

im1 = axes[1].imshow(K_true / K_true[0, 0], cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Matern 3/2 covariance (normalised)  —  ground truth", fontsize=12)
axes[1].set_xticks(range(N_ZONES)); axes[1].set_xticklabels(ZONES, rotation=45, ha="right")
axes[1].set_yticks(range(N_ZONES)); axes[1].set_yticklabels(ZONES)
plt.colorbar(im1, ax=axes[1], label="correlation")

plt.tight_layout()
plt.show()
print(f"SE3–DK2 distance: {D_km[ZONES.index('SE3'), ZONES.index('DK2')]:.0f} km  "
      f"→  correlation {K_true[ZONES.index('SE3'), ZONES.index('DK2')] / K_true[0,0]:.3f}")
print(f"SE1–FI  distance: {D_km[ZONES.index('SE1'), ZONES.index('FI')]:.0f} km  "
      f"→  correlation {K_true[ZONES.index('SE1'), ZONES.index('FI')] / K_true[0,0]:.3f}")

In [ ]:
# ── Generate a short in-memory dataset (2 weeks, 9 zones) ──────────────────
# Used for all downstream demo cells when PostgreSQL is unavailable.

N_HOURS = 24 * 14   # 2 weeks
MEAN_WIND = 7.0

L_true = np.linalg.cholesky(K_true + 1e-6 * np.eye(N_ZONES))
z = rng.standard_normal((N_HOURS, N_ZONES))
wind_raw = MEAN_WIND + (L_true @ z.T).T
wind_raw = np.maximum(wind_raw, 0.0)

# Inject two correlated drought events
for t_idx in [50, 210]:
    affected = rng.choice(N_ZONES, size=5, replace=False)
    wind_raw[t_idx, affected] *= 0.4

timestamps = pd.date_range("2024-01-01", periods=N_HOURS, freq="h")
rows = []
for t_i, ts in enumerate(timestamps):
    for z_i, zid in enumerate(ZONES):
        rows.append({"timestamp": ts, "region_id": zid, "wind_speed_mps": wind_raw[t_i, z_i]})
grid_snapshots_demo = pd.DataFrame(rows)

# Quick visualisation: first 5 zones over time
fig, ax = plt.subplots(figsize=(14, 3.5))
for zid in ZONES[:5]:
    sub = grid_snapshots_demo[grid_snapshots_demo["region_id"] == zid]
    ax.plot(sub["timestamp"], sub["wind_speed_mps"], lw=0.7, label=zid)
ax.axhline(3.0, color="red", lw=1.2, ls="--", label="cut-in threshold (3 m/s)")
ax.set_ylabel("Wind speed (m/s)")
ax.set_title("Synthetic correlated wind speed — 2-week demo window (5 zones)")
ax.legend(fontsize=8, ncol=6)
plt.tight_layout()
plt.show()
print(f"Dataset: {len(grid_snapshots_demo):,} rows  |  {grid_snapshots_demo['region_id'].nunique()} zones  |  {N_HOURS} timesteps")

## Feature Engineering

The raw dataset (`timestamp`, `region_id`, `wind_speed_mps`) needs to be transformed into a 2-column numerical input matrix $X \in \mathbb{R}^{N \times 2}$ that the HSGP model can consume.

### Two input dimensions

| Column | Raw form | Normalised form | Why this scale |
|---|---|---|---|
| Spatial | Integer zone index 0–8 | `index / (n_zones - 1)` → [0, 1] | HSGP basis functions live on `[-c, c]`; unnormalised indices outside this domain get zero spectral energy |
| Temporal | Hours since first observation | `hours / 168` | 168 h = 1 week; normalising by weekly scale concentrates GP mass on the sub-weekly variation we care about |

### Why great-circle (haversine) distance, not Euclidean?

Europe spans ~15° of latitude and ~20° of longitude in the Nordic region. At 60°N, a 1° longitude step is only ~55 km, whereas a 1° latitude step is ~111 km. Euclidean distance in lat/lon space distorts north-south vs east-west separations by 2×. The haversine formula gives exact great-circle distances and makes the Matern kernel dimensionally correct.

### Code (`pipeline/model/feature_engineering.py`)

In [ ]:
from pipeline.model.feature_engineering import (
    ZONE_CENTROIDS as ZONE_CENTROIDS_FE,
    build_spatial_distance_matrix,
    prepare_hsgp_2d_input,
)

# ── Pairwise distance matrix ────────────────────────────────────────────────
zone_ids_fe = list(ZONE_CENTROIDS_FE.keys())
D_fe = build_spatial_distance_matrix(zone_ids_fe)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(D_fe, cmap="YlOrRd")
ax.set_xticks(range(len(zone_ids_fe))); ax.set_xticklabels(zone_ids_fe, rotation=45, ha="right")
ax.set_yticks(range(len(zone_ids_fe))); ax.set_yticklabels(zone_ids_fe)
ax.set_title("Haversine distance matrix (km)", fontsize=12)
plt.colorbar(im, ax=ax, label="km")
plt.tight_layout()
plt.show()

# ── Prepare 2D input for the HSGP ──────────────────────────────────────────
X_demo, y_demo, meta_demo = prepare_hsgp_2d_input(grid_snapshots_demo)

print(f"X shape: {X_demo.shape}   (N_obs x 2)")
print(f"y shape: {y_demo.shape}   (N_obs,)")
print(f"Zones:   {meta_demo['zone_ids']}")
print(f"Spatial column range:   [{X_demo[:, 0].min():.3f}, {X_demo[:, 0].max():.3f}]")
print(f"Temporal column range:  [{X_demo[:, 1].min():.3f}, {X_demo[:, 1].max():.3f}]")

In [ ]:
# ── Visualise the 2D input space ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sc = axes[0].scatter(X_demo[:, 1], X_demo[:, 0], c=y_demo, cmap="RdYlBu_r",
                     s=3, alpha=0.6, vmin=0, vmax=14)
axes[0].set_xlabel("Normalised time (hours / 168)")
axes[0].set_ylabel("Normalised spatial index")
axes[0].set_title("HSGP input space: colour = wind speed (m/s)")
plt.colorbar(sc, ax=axes[0], label="m/s")

axes[1].hist(y_demo, bins=40, color="steelblue", edgecolor="white", lw=0.3)
axes[1].axvline(3.0, color="red", lw=1.5, ls="--", label="cut-in threshold")
axes[1].set_xlabel("Wind speed (m/s)")
axes[1].set_ylabel("Count")
axes[1].set_title("Wind speed distribution — demo dataset")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Fraction below cut-in (3 m/s): {(y_demo < 3.0).mean():.3f}")

## Why Gaussian Processes?

A Gaussian Process (GP) is a probability distribution over *functions*, not over scalars or vectors. Instead of fitting a single best-fit curve, a GP maintains a full posterior distribution over all functions consistent with the data.

### Intuition

Suppose you observe wind speed $y_1, y_2, \ldots, y_N$ at locations (space-time points) $x_1, \ldots, x_N$. A GP says:

> Any finite collection of function values $(f(x_1), \ldots, f(x_N))$ is jointly Gaussian, with a mean function $\mu(x)$ and a covariance function (kernel) $k(x, x')$ that encodes how correlated the function values are.

This is exactly what we need:
- $f(x)$ is the latent "true" wind speed at space-time coordinate $x$
- $k(x, x')$ encodes that nearby zones at similar times are correlated — the thing deterministic models can't capture
- Posterior uncertainty over $f$ propagates through to P(shortfall), giving us honest probabilistic forecasts

### The kernel as domain knowledge

We use a **Matern 5/2** kernel with separate length scales per spatial and temporal dimension (ARD = Automatic Relevance Determination):

$$k(\mathbf{x}, \mathbf{x}') = \eta^2 \cdot k_{\text{M52}}\!\left(\sqrt{\frac{(x_s - x'_s)^2}{\ell_s^2} + \frac{(x_t - x'_t)^2}{\ell_t^2}}\right)$$

The Matern 5/2 kernel is:
$$k_{\text{M52}}(r) = \left(1 + \sqrt{5}\,r + \frac{5}{3}r^2\right)\exp(-\sqrt{5}\,r)$$

- **$\ell_s$** (spatial length scale): controls how quickly spatial correlation decays — we put a prior centred on 50–200 km, matching typical synoptic weather system scales
- **$\ell_t$** (temporal length scale): controls how quickly temporal correlation decays — we put a prior centred on multi-day to weekly timescales
- **$\eta$** (amplitude): the standard deviation of wind speed variation unexplained by the mean
- Matern 5/2 is chosen over RBF because it allows functions that are twice mean-square differentiable — smooth but not infinitely so, matching physical wind fields

### Why not just use a neural network?

A GP gives us:
1. **Exact posterior uncertainty** — not just a point estimate but a full distribution
2. **Interpretable hyperparameters** — $\ell_s$ has a direct physical interpretation as spatial correlation length
3. **Data efficiency** — with 5 years of hourly data we have ~50K points, enough for a GP but barely enough for a large neural network to learn reliable tail behaviour
4. **Calibrated probabilities** — P(shortfall) from a GP posterior is statistically calibrated; from a neural network it requires careful post-hoc calibration

## Why HSGP, Not an Exact GP?

### The O(N³) problem

An exact GP requires computing and inverting the $N \times N$ covariance matrix $K$. For our full dataset:

- 5 years × 8,760 hours/year × 9 zones = **~394,200 observations**
- $394200^3 \approx 6 \times 10^{16}$ floating-point operations — a month of compute time on a modern GPU
- Memory: a 394K × 394K float64 matrix = **~1.2 TB** RAM

This is completely infeasible.

### The HSGP approximation: O(N·m²)

The **Hilbert Space GP (HSGP)** approximation (Solin & Särkkä 2020, Riutort-Mayol et al. 2023) replaces the exact GP with a spectral expansion:

$$f(\mathbf{x}) \approx \sum_{j=1}^{m} \phi_j(\mathbf{x}) \cdot S(\sqrt{\lambda_j})^{1/2} \cdot \beta_j$$

where:
- $\phi_j(\mathbf{x})$ are the **Laplacian eigenfunctions** of the domain $[-c, c]^d$ — these are fixed sinusoids, independent of the kernel hyperparameters
- $\lambda_j$ are the corresponding eigenvalues
- $S(\cdot)$ is the **spectral density** of the kernel (Matern 5/2's Fourier transform) — this is where the length scale $\ell$ enters
- $\beta_j \sim \mathcal{N}(0, 1)$ are the basis coefficients — the actual MCMC parameters

The key insight: $\Phi$ (the $N \times m$ matrix of eigenfunctions) is **computed once** and held constant during sampling. MCMC only needs to update $m = m_s \times m_t$ scalar weights $\beta$ and 4 hyperparameters. The computational cost per gradient evaluation drops from $O(N^3)$ to $O(N \cdot m^2)$.

### Parameter choices

| Parameter | Value | Meaning |
|---|---|---|
| `m_spatial` | 5 | Number of spatial basis functions |
| `m_temporal` | 10 | Number of temporal basis functions |
| `m` total | 50 | 5 × 10 tensor product expansion |
| `c` | 1.5 | Boundary factor: domain is `[-1.5, 1.5]` in normalised coords |

The boundary factor `c` must satisfy $c > 1$; larger values increase approximation quality but reduce the effective resolution. `c = 1.5` is the standard recommendation for normalised inputs on [0, 1].

> **Rule of thumb**: Start with m = 20 total basis functions for exploration. Increase until posterior predictive checks stabilise. For production we use m = 50 (5 × 10).

In [ ]:
# ── Visualise HSGP basis functions on the spatial dimension ─────────────────
# phi_j(x) = (1/sqrt(L)) * sin(pi * j * (x + L) / (2 * L))
# where L = c * x_range (here c=1.5, x in [0,1] so L=1.5)

c = 1.5
L_domain = c  # domain is [-c, c] but inputs normalised to [0,1] so effective range is [0, 1] inside [-1.5, 1.5]

x_plot = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(10, 3.5))

colors = plt.cm.tab10(np.linspace(0, 1, 6))
for j in range(1, 7):
    phi_j = (1.0 / np.sqrt(L_domain)) * np.sin(np.pi * j * (x_plot + L_domain) / (2 * L_domain))
    ax.plot(x_plot, phi_j, label=f"$\\phi_{j}$", color=colors[j - 1], lw=1.5)

ax.set_xlabel("Normalised spatial coordinate (zone index / 8)")
ax.set_ylabel("Basis function value")
ax.set_title("First 6 Laplacian eigenfunctions — HSGP spatial basis")
ax.legend(ncol=6, fontsize=9)
ax.axhline(0, color="black", lw=0.5)
plt.tight_layout()
plt.show()
print("The GP is approximated as a weighted sum of these sinusoids.")
print("Higher j = faster oscillation = capturing finer-scale spatial variation.")
print(f"Total basis functions (m_s=5 × m_t=10): 50 parameters instead of N×N matrix")

## Prior Specification

Before touching data, a Bayesian model requires specifying prior distributions over all parameters. We use **weakly informative priors** — distributions wide enough not to exclude plausible values but narrow enough to regularise inference.

### Hyperparameters and their priors

| Parameter | Prior | Support | Rationale |
|---|---|---|---|
| `ell_spatial` | `InverseGamma(3, 0.5)` | (0, ∞) | Places bulk of mass in [0.1, 0.6] (normalised units), corresponding to 80–480 km — spans from sub-zone to multi-national scales |
| `ell_temporal` | `InverseGamma(3, 1.0)` | (0, ∞) | Places bulk of mass in [0.2, 1.2] (× 168 h), corresponding to 1–8 days — captures synoptic weather timescales |
| `eta` | `HalfNormal(3)` | [0, ∞) | GP amplitude; 95% prior mass below ~6 m/s, consistent with observed ±4 m/s wind deviations |
| `sigma` | `HalfNormal(2)` | [0, ∞) | Observation noise; 95% prior mass below ~4 m/s, allowing for sensor noise and sub-hourly variability |

### Why `InverseGamma` for length scales?

`InverseGamma(alpha, beta)` is the conjugate prior for Gaussian variances. For length scales it has an important practical advantage: it has **zero mass at zero**, preventing the sampler from exploring degenerate near-zero length scales (which cause numerical instability and correspond to white-noise — no spatial correlation). The mode is at $\beta / (\alpha + 1)$ and has heavier tails than a HalfNormal, allowing for longer-range correlations.

In [ ]:
# ── Prior predictive check ───────────────────────────────────────────────────
# Visualise what the priors imply about wind speed before seeing any data.

if not HAS_PYMC:
    print("PyMC not installed — skipping prior predictive check")
else:
    from pipeline.model.hsgp_model import build_hsgp_model

    # Use a very small slice (50 points) for fast prior sampling
    idx_slice = np.random.choice(len(X_demo), size=50, replace=False)
    X_small = X_demo[idx_slice]
    y_small = y_demo[idx_slice]

    model_prior, _ = build_hsgp_model(X_small, y_small, m_spatial=3, m_temporal=5)

    with model_prior:
        prior_pred = pm.sample_prior_predictive(draws=300, random_seed=42)

    prior_y = prior_pred.prior_predictive["y_obs"].values.flatten()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(prior_y, bins=60, density=True, color="#4C72B0", alpha=0.7, label="prior predictive")
    axes[0].hist(y_small, bins=20, density=True, color="coral", alpha=0.7, label="observed data")
    axes[0].axvline(3.0, color="red", ls="--", lw=1.5, label="cut-in threshold")
    axes[0].set_xlabel("Wind speed (m/s)")
    axes[0].set_title("Prior predictive vs observed")
    axes[0].legend()
    axes[0].set_xlim(-5, 20)

    # Prior distributions for hyperparameters
    ell_s_samples = prior_pred.prior["ell_spatial"].values.flatten()
    ell_t_samples = prior_pred.prior["ell_temporal"].values.flatten()
    eta_samples   = prior_pred.prior["eta"].values.flatten()

    axes[1].hist(ell_s_samples, bins=40, alpha=0.6, label=r"$\ell_{spatial}$", density=True)
    axes[1].hist(ell_t_samples, bins=40, alpha=0.6, label=r"$\ell_{temporal}$", density=True)
    axes[1].hist(eta_samples,   bins=40, alpha=0.6, label=r"$\eta$ (amplitude)", density=True)
    axes[1].set_xlabel("Normalised units")
    axes[1].set_title("Prior distributions over hyperparameters")
    axes[1].legend()
    axes[1].set_xlim(0, 5)

    plt.tight_layout()
    plt.show()
    print(f"Prior predictive range: [{prior_y.min():.1f}, {prior_y.max():.1f}] m/s")
    print("The prior covers the full range of observed wind speeds — weakly informative, not too tight.")

## Full Model Code — Annotated

The complete model lives in `pipeline/model/hsgp_model.py`. Here is every line with a detailed explanation of *why* it is written that way.

```python
with pm.Model(coords=coords) as model:
```
The `coords` dict (`{"obs": np.arange(n_obs)}`) maps dimension names to coordinate labels. This makes ArviZ output human-readable — instead of `y_obs[0]` through `y_obs[N]` you get named indices. Required for `dims=` to work on the likelihood.

```python
    X_data = pm.Data("X", X)
    y_data = pm.Data("y", y)
```
`pm.Data` wraps arrays in PyTensor shared variables. The advantage: at prediction time you call `pm.set_data({"X": X_new})` without rebuilding the model. This is how the `tail_risk_forecasts` asset generates out-of-sample predictions for 24h/48h/72h horizons.

```python
    ell_spatial = pm.InverseGamma("ell_spatial", alpha=3.0, beta=0.5)
    ell_temporal = pm.InverseGamma("ell_temporal", alpha=3.0, beta=1.0)
    eta = pm.HalfNormal("eta", sigma=3.0)
    sigma = pm.HalfNormal("sigma", sigma=2.0)
```
Four scalar MCMC parameters. The sampler only has to explore a 4-dimensional hyperparameter space plus the `m=50` basis coefficients $\beta$ — a total of 54 parameters regardless of dataset size N.

```python
    cov = eta**2 * pm.gp.cov.Matern52(2, ls=[ell_spatial, ell_temporal])
```
A single 2D Matern 5/2 kernel with ARD length scales. The `2` is the input dimensionality. `ls=[ell_spatial, ell_temporal]` implements **separable** covariance: the kernel treats spatial and temporal dimensions with different length scales but uses the same functional form. This is an approximation to a fully separable product kernel, but one that HSGP's single-kernel requirement forces — and it works well in practice.

```python
    gp = pm.gp.HSGP(m=[m_spatial, m_temporal], c=1.5, cov_func=cov)
    f = gp.prior("f", X=X_data)
```
`pm.gp.HSGP` builds the basis function expansion. `m=[5, 10]` means 5 spatial basis functions × 10 temporal = 50 total. Internally this computes $\Phi$ (the eigenfunction matrix) at `X_data` and adds 50 latent `beta` variables $\sim \mathcal{N}(0, 1)$. Calling `gp.prior("f", X=...)` returns a deterministic variable $f = \Phi \text{diag}(S)^{1/2} \beta$.

```python
    pm.Normal("y_obs", mu=f, sigma=sigma, observed=y_data, dims="obs")
```
Gaussian likelihood. Wind speed is continuous and roughly symmetric around the mean, so Gaussian noise is appropriate. `sigma` captures both measurement noise and sub-hourly variability not explained by the GP.

### Model DAG

In [ ]:
# ── Build model and render graphviz DAG ─────────────────────────────────────

if not HAS_PYMC:
    print("PyMC not installed — skipping model build")
else:
    from pipeline.model.hsgp_model import build_hsgp_model

    idx_slice = np.random.choice(len(X_demo), size=80, replace=False)
    X_small = X_demo[idx_slice]
    y_small = y_demo[idx_slice]

    model_demo, gp_demo = build_hsgp_model(X_small, y_small, m_spatial=3, m_temporal=5)

    print(model_demo)
    print()
    print("Initial log-probabilities (sanity check — should not be -inf or NaN):")
    print(model_demo.point_logps())

    try:
        graph = pm.model_to_graphviz(model_demo)
        display(graph)
    except Exception as e:
        print(f"Graphviz render failed (likely graphviz not installed): {e}")
        print("Model DAG: ell_spatial, ell_temporal, eta, sigma → cov → gp.prior(f) → y_obs")

## MCMC Inference with Nutpie

### The NUTS algorithm

No-U-Turn Sampling (NUTS) is a variant of Hamiltonian Monte Carlo (HMC) that automatically tunes the trajectory length. HMC simulates Hamiltonian dynamics — it treats the log-posterior as a potential energy and samples trajectories through parameter space using gradient information. This allows it to make large, low-correlation steps in high-dimensional spaces, unlike random-walk Metropolis-Hastings.

For a 54-dimensional model (4 hyperparameters + 50 basis coefficients), NUTS typically needs 100–500 tuning steps to learn the mass matrix and converges in 200–500 posterior draws per chain.

### Why Nutpie?

[Nutpie](https://github.com/pymc-devs/nutpie) is a Rust reimplementation of the NUTS sampler that PyMC calls via a Python binding. It:
- Runs **5–10× faster** than PyMC's default Python-based NUTS (PyTensor backend)
- Computes gradients more efficiently using a compiled C extension
- Supports all standard PyMC models including GPs, hierarchical models, and custom likelihoods
- Falls back cleanly: if `import nutpie` fails, we use `nuts_sampler="pymc"` (slower but correct)

### Sampling configuration

```python
idata = pm.sample(
    draws=300,           # posterior draws per chain (saved to idata.posterior)
    tune=200,            # warm-up/tuning steps (discarded) for mass matrix adaptation
    chains=2,            # independent chains for convergence diagnostics
    nuts_sampler="nutpie",
    random_seed=42,
    idata_kwargs={"log_likelihood": True},  # nutpie ignores this silently — compute manually
)
idata.to_netcdf("results.nc")  # Save immediately — post-MCMC crash loses everything!
```

**Important**: Nutpie silently ignores `idata_kwargs={"log_likelihood": True}`. If you need LOO-CV, compute it manually afterwards:
```python
pm.compute_log_likelihood(idata, model=model)
```

### What gets stored in InferenceData

| Group | Contents |
|---|---|
| `posterior` | MCMC samples for all parameters: `ell_spatial`, `ell_temporal`, `eta`, `sigma`, `f` |
| `sample_stats` | Diagnostics per step: `diverging`, `tree_depth`, `energy`, `accept` |
| `observed_data` | The `y` array passed as `observed=` |
| `constant_data` | The `X` data container |
| `posterior_predictive` | Filled in by `pm.sample_posterior_predictive()` after sampling |

In [ ]:
# ── Run MCMC on the demo dataset ─────────────────────────────────────────────
# Uses a small slice (100 points) to keep runtime under ~60 s.
# Production runs use the full dataset (~394K points) on KubeRay.

if not HAS_PYMC:
    print("PyMC not installed — skipping MCMC")
    idata_demo = None
else:
    try:
        import nutpie
        nuts_sampler = "nutpie"
        print("Using nutpie (Rust NUTS)")
    except ImportError:
        nuts_sampler = "pymc"
        print("nutpie not found — using PyMC default NUTS (slower)")

    idx_slice = np.random.default_rng(0).choice(len(X_demo), size=150, replace=False)
    X_fit = X_demo[idx_slice]
    y_fit = y_demo[idx_slice]

    model_fit, gp_fit = build_hsgp_model(X_fit, y_fit, m_spatial=3, m_temporal=5)

    with model_fit:
        idata_demo = pm.sample(
            draws=300,
            tune=200,
            chains=2,
            nuts_sampler=nuts_sampler,
            random_seed=42,
            progressbar=True,
        )

    print("\nSampling complete.")
    print(f"Posterior shape: {idata_demo.posterior['ell_spatial'].shape}  (chains, draws)")

## Diagnostics

We follow a strict 4-phase diagnostic workflow (via `pipeline/model/diagnostics.py` + ArviZ). **Never interpret model parameters until all phases pass.**

### Pass criteria

| Diagnostic | Pass threshold | What failure means |
|---|---|---|
| Divergences | 0 (< 0.1% acceptable) | Sampler stuck in curved geometry; non-centred parameterisation or tighter priors needed |
| R-hat | < 1.01 for all params | Chains haven't mixed; need more tuning or the model is unidentified |
| ESS bulk | > 400 | Too few effective samples; need more draws or reduce autocorrelation |
| ESS tail | > 400 | Tail of posterior not well explored |

### Phase 1: Immediate convergence checks

In [ ]:
# ── Phase 1: Divergences + summary ──────────────────────────────────────────

if idata_demo is None:
    print("No idata — skipping diagnostics")
else:
    from pipeline.model.diagnostics import run_mcmc_diagnostics

    diag = run_mcmc_diagnostics(idata_demo, var_names=["ell_spatial", "ell_temporal", "eta", "sigma"])
    print("=== MCMC Diagnostic Summary ===")
    for k, v in diag.items():
        symbol = "PASS" if (
            (k == "n_divergences" and v == 0) or
            (k == "max_r_hat" and v < 1.01) or
            ("ess" in k and v > 400)
        ) else "WARN"
        print(f"  [{symbol}] {k}: {v:.4g}")

    print()
    summary = az.summary(
        idata_demo,
        var_names=["ell_spatial", "ell_temporal", "eta", "sigma"],
    )
    print(summary[["mean", "sd", "hdi_3%", "hdi_97%", "ess_bulk", "ess_tail", "r_hat"]])

In [ ]:
# ── Phase 2: Trace plots (visual convergence) ────────────────────────────────

if idata_demo is None:
    print("No idata — skipping trace plots")
else:
    az.plot_trace(
        idata_demo,
        var_names=["ell_spatial", "ell_temporal", "eta", "sigma"],
        compact=True,
        figsize=(12, 6),
    )
    plt.suptitle("Trace plots — left: posterior density per chain, right: sample trajectory", y=1.02)
    plt.tight_layout()
    plt.show()
    print("Good mixing: densities overlap between chains (left); trajectories look like white noise (right)")

In [ ]:
# ── Phase 2: Energy diagnostic ───────────────────────────────────────────────
# The energy transition distribution (marginal) should match the energy distribution.
# A large gap indicates HMC is not exploring the posterior efficiently.

if idata_demo is None:
    print("No idata")
else:
    az.plot_energy(idata_demo, figsize=(8, 3))
    plt.title("Energy distribution — marginal vs transition")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Phase 3: Posterior predictive check ─────────────────────────────────────
# Does the model reproduce the distribution of observed wind speeds?

if idata_demo is None:
    print("No idata")
else:
    with model_fit:
        pm.sample_posterior_predictive(idata_demo, extend_inferencedata=True, random_seed=42)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    az.plot_ppc(idata_demo, kind="kde", ax=axes[0], num_pp_samples=100)
    axes[0].set_title("Posterior predictive check — density")
    axes[0].set_xlabel("Wind speed (m/s)")

    az.plot_ppc(idata_demo, kind="cumulative", ax=axes[1], num_pp_samples=100)
    axes[1].set_title("Posterior predictive check — CDF")
    axes[1].set_xlabel("Wind speed (m/s)")

    plt.tight_layout()
    plt.show()
    print("Dark line = observed data  |  Light lines = posterior predictive samples")
    print("Observed should fall within the predictive envelope for a well-calibrated model.")

In [ ]:
# ── Phase 4: Parameter posteriors ────────────────────────────────────────────

if idata_demo is None:
    print("No idata")
else:
    az.plot_posterior(
        idata_demo,
        var_names=["ell_spatial", "ell_temporal", "eta", "sigma"],
        figsize=(13, 3.5),
    )
    plt.suptitle("Posterior distributions for HSGP hyperparameters", y=1.02)
    plt.tight_layout()
    plt.show()

    # Convert normalised length scales back to physical units
    ell_s_post = idata_demo.posterior["ell_spatial"].values.flatten()
    ell_t_post = idata_demo.posterior["ell_temporal"].values.flatten()
    print(f"ell_spatial posterior mean: {ell_s_post.mean():.3f} (normalised)")
    print(f"  → spatial correlation ~halved at: {ell_s_post.mean() * 8:.1f} zone-steps")
    print(f"ell_temporal posterior mean: {ell_t_post.mean():.3f} (normalised)")
    print(f"  → temporal correlation ~halved at: {ell_t_post.mean() * 168:.0f} hours")

## Spatial Correlation Recovery

A key validation: the HSGP model should *recover* the ground-truth spatial correlation structure that the generator encoded. We compare:
- **True covariance** (from the generator's Matern 3/2 kernel, `sigma2=4`, `ell=100 km`)
- **Learned correlation** (derived from the HSGP posterior predictive samples across zones)

The Frobenius norm of the difference quantifies fit quality and is logged to MLflow as `spatial_corr_norm`.

In [ ]:
# ── Compare true covariance vs learned correlation ───────────────────────────

if idata_demo is None:
    print("No idata — showing true covariance only")
    K_norm = K_true / K_true[0, 0]
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(K_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(N_ZONES)); ax.set_xticklabels(ZONES, rotation=45, ha="right")
    ax.set_yticks(range(N_ZONES)); ax.set_yticklabels(ZONES)
    ax.set_title("True Matern 3/2 covariance (normalised)")
    plt.colorbar(im, ax=ax)
    plt.tight_layout(); plt.show()
else:
    # Extract posterior predictive samples; compute empirical correlation per zone-pair
    pp_samples = idata_demo.posterior_predictive["y_obs"].values  # (chains, draws, n_obs)
    pp_flat = pp_samples.reshape(-1, pp_samples.shape[-1])         # (total_draws, n_obs)

    # Build zone × time matrix from the sampled observations
    zone_ids_fit = meta_demo["zone_ids"]
    n_zones_fit = len(zone_ids_fit)
    n_times_fit = len(X_fit) // n_zones_fit if len(X_fit) % n_zones_fit == 0 else None

    if n_times_fit is None:
        print("Irregular zone coverage — using hyperparameter correlation as proxy")
        ell_s_flat = idata_demo.posterior["ell_spatial"].values.flatten()
        ell_t_flat = idata_demo.posterior["ell_temporal"].values.flatten()
        eta_flat   = idata_demo.posterior["eta"].values.flatten()
        C_learned  = np.corrcoef(np.vstack([ell_s_flat, ell_t_flat, eta_flat]))
        print("Hyperparameter correlation matrix:")
        print(C_learned)
    else:
        # Reshape posterior predictive to (total_draws, n_zones, n_times)
        pp_zones = pp_flat.reshape(-1, n_zones_fit, n_times_fit)
        # Mean across draws → (n_zones, n_times); then compute inter-zone correlation
        pp_mean  = pp_zones.mean(axis=0)  # (n_zones, n_times)
        C_learned = np.corrcoef(pp_mean)

        K_norm = K_true[:n_zones_fit, :n_zones_fit] / K_true[0, 0]

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        im0 = axes[0].imshow(K_norm, cmap="Blues", vmin=0, vmax=1)
        axes[0].set_title("True Matern 3/2\n(generator ground truth)")
        axes[0].set_xticks(range(n_zones_fit)); axes[0].set_xticklabels(zone_ids_fit, rotation=45, ha="right")
        axes[0].set_yticks(range(n_zones_fit)); axes[0].set_yticklabels(zone_ids_fit)
        plt.colorbar(im0, ax=axes[0])

        im1 = axes[1].imshow(C_learned, cmap="Blues", vmin=0, vmax=1)
        axes[1].set_title("HSGP learned correlation\n(posterior predictive mean)")
        axes[1].set_xticks(range(n_zones_fit)); axes[1].set_xticklabels(zone_ids_fit, rotation=45, ha="right")
        axes[1].set_yticks(range(n_zones_fit)); axes[1].set_yticklabels(zone_ids_fit)
        plt.colorbar(im1, ax=axes[1])

        diff = np.abs(K_norm - C_learned)
        im2 = axes[2].imshow(diff, cmap="Reds", vmin=0)
        axes[2].set_title("Absolute difference")
        axes[2].set_xticks(range(n_zones_fit)); axes[2].set_xticklabels(zone_ids_fit, rotation=45, ha="right")
        axes[2].set_yticks(range(n_zones_fit)); axes[2].set_yticklabels(zone_ids_fit)
        plt.colorbar(im2, ax=axes[2])

        frob = np.linalg.norm(K_norm - C_learned, "fro")
        plt.suptitle(f"Spatial correlation recovery  |  Frobenius distance = {frob:.4f}", fontsize=12)
        plt.tight_layout()
        plt.show()
        print(f"Frobenius norm of difference: {frob:.4f}  (lower = better recovery of spatial structure)")

## Tail Risk: P(Wind Shortfall)

### The physical threshold

Most modern wind turbines have a **cut-in wind speed** of 3–4 m/s. Below this, blades do not rotate fast enough for the generator to produce power. We use **3.0 m/s** as the shortfall threshold:

$$P(\text{shortfall})_z = P(\text{wind}_z < 3.0 \text{ m/s})$$

### How we compute it

1. After sampling the HSGP posterior, we call `gp.conditional("fcond", Xnew=X_new)` to extend the GP to new (unseen) space-time coordinates — specifically, the 9 zones at +24h, +48h, +72h ahead of the last observation.
2. We draw 500 posterior predictive samples of wind speed at each future location using `pm.sample_posterior_predictive()`.
3. For each zone and horizon, `P(shortfall) = fraction of 500 samples < 3.0 m/s`.

The result is a DataFrame with columns `(timestamp, region_id, horizon_h, p_shortfall)` that the FastAPI service reads from the Gold bucket.

### Connection to the API

```
GET /forecast/SE3?horizon_h=48
→ {"region_id": "SE3", "horizon_h": 48, "p_shortfall": 0.14, "status": "ok"}
```

A `p_shortfall` above the alert threshold (default 0.9) triggers `risk_alerts` in Dagster.

In [ ]:
# ── Compute tail-risk forecasts for 24h / 48h / 72h horizons ────────────────

if idata_demo is None:
    print("No idata — generating illustrative tail-risk from prior")
    # Synthetic illustration
    horizons = [24, 48, 72]
    p_shortfall_demo = rng.beta(0.5, 5, (len(ZONES), len(horizons)))
    p_shortfall_demo[:3, :] *= 0.5  # SE zones: lower risk
    p_shortfall_demo[6:8, :] *= 1.5  # DK zones: higher risk
    p_shortfall_demo = np.clip(p_shortfall_demo, 0, 1)
else:
    from pipeline.model.hsgp_model import sample_posterior_predictive, compute_tail_risk

    zone_ids_fit = meta_demo["zone_ids"]
    n_zones_fit  = len(zone_ids_fit)
    zone_to_idx  = {z: i for i, z in enumerate(zone_ids_fit)}

    t_max_norm  = X_fit[:, 1].max()
    result_tmp  = {"idata": idata_demo, "model": model_fit, "gp": gp_fit}
    horizons    = [24, 48, 72]
    p_shortfall_demo = np.zeros((n_zones_fit, len(horizons)))

    for h_idx, horizon_h in enumerate(horizons):
        horizon_norm = horizon_h / 168.0
        sp_new = np.array([zone_to_idx[z] / max(n_zones_fit - 1, 1) for z in zone_ids_fit])
        tp_new = np.full(n_zones_fit, t_max_norm + horizon_norm)
        samples = sample_posterior_predictive(result_tmp, sp_new, tp_new, n_samples=500)
        p_shortfall_demo[:, h_idx] = compute_tail_risk(samples, threshold_mps=3.0)

# ── Plot results ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
colors = ["#2196F3", "#FF9800", "#F44336"]

zone_labels = ZONES[:len(p_shortfall_demo)]
for h_idx, horizon_h in enumerate(horizons):
    ax = axes[h_idx]
    bars = ax.barh(zone_labels, p_shortfall_demo[:, h_idx], color=colors[h_idx], alpha=0.8)
    ax.axvline(0.1, color="orange", lw=1.5, ls="--", label="10% threshold")
    ax.axvline(0.9, color="red",    lw=1.5, ls="--", label="alert threshold")
    ax.set_xlim(0, 1)
    ax.set_xlabel("P(shortfall)")
    ax.set_title(f"+{horizon_h}h horizon", fontsize=12)
    if h_idx == 0:
        ax.legend(fontsize=8)

plt.suptitle("P(wind speed < 3 m/s) by zone and forecast horizon", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# Identify at-risk zones
at_risk = []
for h_idx, horizon_h in enumerate(horizons):
    for z_idx, zid in enumerate(zone_labels):
        if p_shortfall_demo[z_idx, h_idx] > 0.1:
            at_risk.append(f"{zid} @ +{horizon_h}h ({p_shortfall_demo[z_idx, h_idx]:.2f})")
if at_risk:
    print("Zones with P(shortfall) > 10%:")
    for r in at_risk:
        print(f"  {r}")
else:
    print("No zones exceed the 10% shortfall threshold in this window.")

## The Dagster Pipeline

In the MVP (local/Kubernetes single-node), the full workflow is orchestrated as a **Dagster asset graph** in `pipeline/dagster/assets.py`. Each asset is a Python function decorated with `@asset`; Dagster tracks dependencies, schedules re-execution, and manages IO through pluggable IO managers.

### Asset graph

```
raw_weather_obs
     │  (reads from PostgreSQL weather_obs table)
     │  (IO manager: MinIO Parquet)
     ▼
grid_snapshots
     │  (1-hour tumbling window aggregation; one row per (timestamp, region_id))
     │  (IO manager: MinIO Parquet)
     ▼
hsgp_model ─────────────────────────────────────┐
     │  (train_hsgp() with Nutpie NUTS)          │
     │  (saves idata.nc to MinIO Gold)            │
     │  (IO manager: MinIO Pickle for metadata)  │
     ▼                                            │
tail_risk_forecasts  ◄───────────────────────────┘
     │  (sample_posterior_predictive for 24/48/72h)
     │  (compute_tail_risk: P(wind < 3 m/s) per zone)
     │  (IO manager: MinIO Parquet)
     ▼
risk_alerts
     (log/publish alerts for zones where p_shortfall ≥ 0.9)
     (Phase 2: publish to Redpanda topic)
```

### Schedules

| Schedule | Cron | Default | Purpose |
|---|---|---|---|
| `grid_resilience_retrain` | `0 */6 * * *` | **RUNNING** | Retrain HSGP on latest 6 hours of data |
| `grid_resilience_full_pipeline` | `0 6 * * *` | STOPPED | Full pipeline from raw data; daily |

### IO managers

| Manager key | Format | Usage |
|---|---|---|
| `minio_io_manager` | Parquet | DataFrames (weather, snapshots, forecasts) |
| `minio_pickle_io_manager` | Pickle | Model metadata dicts |
| `noop_io_manager` | — | `risk_alerts` (side-effects only, no output stored) |

In production (KubeRay), the Dagster assets are replaced by `inference/run_inference.py` running as a RayJob, reading from the Silver bucket and writing to Gold.

## MLflow Experiment Tracking

Every training run is logged to an MLflow experiment called `grid-resilience-hsgp` via `pipeline/model/mlflow_utils.py`. This gives us a reproducible audit trail of every model version.

### What gets logged per run

| Category | Key | Value |
|---|---|---|
| **Params** | `m_spatial` | 5 (basis functions per spatial dimension) |
| **Params** | `m_temporal` | 10 (basis functions per temporal dimension) |
| **Params** | `n_zones` | 9 |
| **Params** | `n_iter` | 300 (draws per chain) |
| **Metrics** | `n_divergences` | 0 = healthy |
| **Metrics** | `max_r_hat` | < 1.01 = converged |
| **Metrics** | `min_ess_bulk` | > 400 = sufficient samples |
| **Metrics** | `crps` | MAE of median forecast (lower = better) |
| **Metrics** | `spatial_corr_norm` | Frobenius norm of correlation matrix |
| **Artifacts** | `spatial_correlation/*.npy` | Learned correlation matrix |
| **Artifacts** | `metadata/*.json` | Zone IDs, n_zones, n_timesteps |
| **Tags** | `model_type` | `hsgp` |
| **Tags** | `kernel` | `matern_separable` |

In [ ]:
# ── Demonstrate MLflow logging (without a running server) ───────────────────
# Set MLFLOW_TRACKING_URI to http://localhost:5000 to log to a real server.
# With no server, we use a local ./mlruns directory.

import os
os.environ.setdefault("MLFLOW_TRACKING_URI", "./mlruns")

try:
    import mlflow
    from pipeline.model.diagnostics import run_diagnostics
    from pipeline.model.mlflow_utils import log_hsgp_run

    if idata_demo is not None:
        diag_full = run_diagnostics(
            None,
            idata=idata_demo,
            observed=y_fit,
            posterior_predictive=(
                idata_demo.posterior_predictive["y_obs"].values.reshape(-1, len(y_fit))
                if hasattr(idata_demo, "posterior_predictive") else None
            ),
        )
    else:
        diag_full = {"n_divergences": 0, "max_r_hat": 1.001, "min_ess_bulk": 450.0, "crps": 0.82}

    meta_log = {"zone_ids": ZONES, "n_zones": N_ZONES, "n_timesteps": N_HOURS}

    log_hsgp_run(
        trace_or_guide=None,
        diagnostics=diag_full,
        metadata=meta_log,
        run_name="notebook_demo_run",
        tracking_uri="./mlruns",
        log_artifacts=True,
        m_spatial=3,
        m_temporal=5,
        n_iter=300,
    )
    print("MLflow run logged to ./mlruns")
    print("Metrics logged:", {k: v for k, v in diag_full.items()})
    print()
    print("To inspect: mlflow ui --backend-store-uri ./mlruns")

except ImportError:
    print("mlflow not installed — pip install mlflow")
except Exception as e:
    print(f"MLflow logging failed: {e}")

## What's Next: Phase 2 Roadmap

The current implementation (Phase 1) is a fully functional Bayesian inference pipeline on synthetic data with a local PostgreSQL + MinIO backend. Phase 2 replaces synthetic inputs with real data and adds real-time streaming.

### Real data ingestion

- **ERA5 reanalysis**: `data-engineering/scripts/fetch_era5_nordic.py` fetches 10 m wind speed NetCDF files from the ECMWF CDS API (credentials in `.cdsapirc`). These cover 1940–present at 0.25° × 0.25° grid resolution over the Nordic domain.
- **PySpark ETL**: `data-engineering/processing/spark_era5_ingest.py` runs as a `SparkApplication` on Kubernetes (1 driver + 2 executors). It reads Bronze NetCDF, performs a spatial join to zone centroids, and writes normalised Silver Parquet.
- **ENTSO-E load data**: Real electricity demand and cross-border flow data joins the weather observations to model demand-side risk.

### Real-time streaming (Bytewax + Redpanda)

- `pipeline/streaming/spatial_windowing.py`: A Bytewax 1-hour tumbling window consumes from a Redpanda Kafka topic `weather-obs`, aggregates all zone observations within each window, and emits `grid_snapshots` records directly.
- `pipeline/streaming/anomaly_detection.py`: A 6-hour sliding window computes running cross-zone correlations. When the correlation structure indicates a correlated drought developing (all zones above a cross-correlation threshold simultaneously), it emits an early-warning alert.

### Model scaling

- The KubeRay `RayJob` in `inference/bayesian-grid-model.yaml` distributes posterior sampling across the `ray-kuberay` cluster in the `datasci` namespace.
- With the full 5-year ERA5 dataset (~394K observations), `m = [10, 20]` (200 total basis functions) is more appropriate than the current `m = [5, 10]` used for MVP speed.
- NumPyro backend (`nuts_sampler="numpyro"`) enables GPU acceleration via JAX for even faster sampling on NVIDIA hardware.

### Dashboard

- A Streamlit app reads the `tail_risk_forecasts.parquet` from the Gold bucket and renders a choropleth map of P(shortfall) per zone for each forecast horizon.
- P10/P50/P90 wind speed quantiles are derived from the full posterior predictive distribution.

### ExternalSecrets

- API keys for ECMWF CDS and ENTSO-E are currently in `.env` (local dev only). Phase 2 provisions them as Kubernetes `ExternalSecret` resources backed by a Vault or cloud KMS.

---

## Summary

| Component | Technology | Status |
|---|---|---|
| Synthetic data | Matern Cholesky generator | Done |
| Feature engineering | Haversine + normalisation | Done |
| HSGP model | PyMC v5 `pm.gp.HSGP` + Matern 5/2 | Done |
| MCMC sampler | Nutpie (Rust NUTS) | Done |
| Diagnostics | ArviZ R-hat / ESS / PPC | Done |
| Tail risk | P(wind < 3 m/s) via posterior predictive | Done |
| Pipeline orchestration | Dagster asset graph | Done |
| Experiment tracking | MLflow `grid-resilience-hsgp` | Done |
| REST API | FastAPI `/forecast` + `/correlations` | Done |
| ERA5 real data | ECMWF CDS API + PySpark ETL | Phase 2 |
| Real-time streaming | Bytewax + Redpanda | Phase 2 |
| Streamlit dashboard | P10/P50/P90 choropleth | Phase 2 |
| GPU sampling | NumPyro/JAX backend | Phase 2 |